# Learning Objectives

* Build a metagenome-assembled-genomes (MAG) Pegasus workflow from shell wrappers baked into an Apptainer container.
* Generate the workflow's catalogs and DAG with the same generator class the command-line tool uses.
* Visualize a workflow DAG before submitting it.
* Plan, submit, monitor, and pull statistics for a Pegasus workflow from a notebook.
* Inspect the QC, assembly and gene-prediction outputs produced by a run.

# MAG Workflow

This workflow assembles metagenomes into genomes, equivalent to the
[nf-core/mag](https://github.com/nf-core/mag) Nextflow pipeline. Each sample is read-QC'd, trimmed, assembled and
annotated independently; a final MultiQC report aggregates all samples. See `README.md` in this repository for the
full pipeline description and options.

**Workflow Jobs (per sample, then one aggregate):**
1. `fastqc` - read quality reports
2. `fastp` - adapter and quality trimming
3. `megahit` (or `spades`) - assembly into contigs
4. `quast` - assembly QC
5. `prodigal` - gene prediction
6. `metabat2` / `checkm2` / `gtdbtk` / `prokka` - binning, bin QC, taxonomy, annotation (skipped in this demo: they need large databases)
7. `multiqc` - one report over all samples

**Prerequisites:** [Pegasus WMS](https://pegasus.isi.edu/) >= 5.0, [HTCondor](https://htcondor.org/) >= 10.2,
Python 3.8+ with `pyyaml`, and [Apptainer](https://apptainer.org/) — all assumed to already be available in
this notebook's environment.

## 1. Prepare Sample Input Data

`download_test_data()` (from `workflow_generator.py`, the same function `--test` uses) fetches the two small
paired-end nf-core/mag test samples (~10 MB) into `output/test_data/` and writes a samplesheet for them.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from workflow_generator import MagWorkflow, apply_resource_caps, download_test_data

samples, samplesheet = download_test_data(str(Path("output").resolve()))
print(f"Samples: {[s['id'] for s in samples]}")

## 2. Build the Apptainer Container

The workflow's jobs run inside one Apptainer container (`Apptainer/MAG_Container.def`) that bundles FastQC,
fastp, MEGAHIT, SPAdes, QUAST, Prodigal, MetaBAT2, CheckM2, GTDB-Tk, Prokka and MultiQC, plus the wrapper scripts.
Building it takes a while; skip the rebuild if the `.sif` already exists.

In [ ]:
%%bash
if [ ! -f Apptainer/MAG_Container.sif ]; then
    apptainer build Apptainer/MAG_Container.sif Apptainer/MAG_Container.def
else
    echo "Container already built: Apptainer/MAG_Container.sif"
fi

## 3. Generate the Workflow

The following cell uses the `MagWorkflow` class from `workflow_generator.py` — the same generator used by the
command-line tool — to build the Pegasus workflow DAG (site, transformation, and replica catalogs) and writes it to
`workflow.yml`. It runs the README's quick test: binning, taxonomy and annotation are skipped, and every job is capped
at 14 GB so it fits a 16 GB worker.

`create_sites_catalog()` writes a self-contained site catalog for this machine's HTCondor pool (`local` +
`compute`). To run on a resource with a centrally hosted site catalog
([pegasus-site-catalogs](https://github.com/pegasushub/pegasus-site-catalogs/tree/main/conf), e.g.
`access-pegasus.yml`, `unity.yml`), set `HOSTED_SITE_CATALOG` instead — or set it once in `~/.pegasusrc`.

In [ ]:
DAGFILE = "workflow.yml"
EXEC_SITE = "compute"
HOSTED_SITE_CATALOG = None  # e.g. "unity.yml"; None = local HTCondor catalog below

apply_resource_caps(max_memory_gb=14)

workflow = MagWorkflow(
    samples,
    dagfile=DAGFILE,
    skip_binning=True,
    skip_taxonomy=True,
    skip_annotation=True,
)

print("Creating workflow properties...")
workflow.create_pegasus_properties(hosted_site_catalog=HOSTED_SITE_CATALOG)

if HOSTED_SITE_CATALOG is None:
    print("Creating execution sites...")
    workflow.create_sites_catalog(exec_site_name=EXEC_SITE)

print("Creating transformation catalog...")
workflow.create_transformation_catalog(exec_site_name=EXEC_SITE)

print("Creating replica catalog...")
workflow.create_replica_catalog()

print("Creating MAG workflow DAG...")
workflow.create_workflow()

workflow.write()
print("\nMAG Workflow has been generated!")

## 4. View the Generated Workflow DAG

Before submitting, visualize the workflow DAG using `pegasus-graphviz`. Each sample is a QC → trim → assemble →
QUAST/Prodigal chain; both feed the final MultiQC job.

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image

Image(filename="workflow.png")

## 5. Plan and Submit the Workflow

The workflow will be planned and submitted for execution on the `compute` site defined in the site catalog chosen
above.

The CLI version of this generator (`workflow_generator.py --test ...`) writes the same workflow non-interactively and
prints the `pegasus-plan` command; it never submits by itself. Submitting is this explicit step.

In [ ]:
workflow.plan_submit(exec_site_name=EXEC_SITE)

## Workflow Status Monitoring

After successful submission, monitor the workflow status. The output shows job counts and idle/running/completed
states. `wait()` blocks (with a progress bar) until the workflow completes or fails.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 6. Statistics

Once the workflow completes, pull execution statistics from the Pegasus provenance database.

In [ ]:
workflow.statistics()

## 7. Examine the Results

The workflow stages the following final outputs to `output/` (to `wf-output/` instead when
`HOSTED_SITE_CATALOG` is set: the hosted catalog has no `local` site, so Pegasus uses its default one), per sample:

| Output | Description |
|--------|-------------|
| `{sample}_R1_fastqc.html`, `{sample}_R2_fastqc.html` (+ `.zip`) | FastQC read-quality reports |
| `{sample}_fastp.html`, `{sample}_fastp.json`, `{sample}_trimmed_R{1,2}.fastq.gz` | fastp trimming report and trimmed reads |
| `{sample}_contigs.fa`, `{sample}_assembly.log` | MEGAHIT assembly |
| `{sample}_quast_report.tsv`, `{sample}_quast_report.html` | Assembly QC |
| `{sample}_genes.faa`, `{sample}_genes.gff` | Prodigal gene predictions |
| `multiqc_report.html`, `multiqc_data.json` | Aggregate report over all samples |

In [ ]:
!ls -lt output/

The assembly statistics for each sample:

In [ ]:
import pandas as pd

for s in samples:
    print(f"== {s['id']}")
    print(pd.read_csv(f"output/{s['id']}_quast_report.tsv", sep="\t").to_string(index=False))